In [5]:
# -*- coding: utf-8 -*-
"""
Лабораторная работа №3
Исследование скрытых характеристик сигналов электроэнцефалографии (eeg2.edf)
"""

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as signal
import mne

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["font.size"] = 11

# 1. Определение рабочей папки (файлы лежат рядом с ноутбуком)
WORK_DIR = Path.cwd()  # или задайте явно: Path(r"D:\MASTER\Term4\2\Seminar_GHADEER\Lab03\LAB3")
print("Рабочая папка:", WORK_DIR)

# Пути к необходимым файлам
edf_path = WORK_DIR / "eeg2.edf"
ann_path = WORK_DIR / "annotations_2017_C.csv"
clinical_path = WORK_DIR / "clinical_information.csv"

# Проверка наличия
for f in [edf_path, ann_path, clinical_path]:
    if not f.exists():
        raise FileNotFoundError(f"Файл {f} не найден!")

# 2. Загрузка клинической информации и аннотаций
clinical_df = pd.read_csv(clinical_path)
annotations_df = pd.read_csv(ann_path)

# Определение ID записи eeg2
record_name = "eeg2"
record_row = clinical_df.loc[clinical_df["EEG file"] == record_name]
if record_row.empty:
    raise ValueError(f"Запись {record_name} не найдена в clinical_information.csv")
record_row = record_row.iloc[0]
record_id = int(record_row["ID"])
print(f"\nЗапись: {record_name}, ID = {record_id}")
print(record_row[["ID", "EEG file", "Diagnosis", "Number of Reviewers Annotating Seizure"]])

# Извлечение столбца аннотаций для данного ID
if str(record_id) not in annotations_df.columns:
    raise KeyError(f"Столбец {record_id} отсутствует в файле аннотаций")
ann_series = annotations_df[str(record_id)].fillna(0).astype(int)

# 3. Функция выделения непрерывных интервалов с меткой "1" (приступ)
def get_intervals(binary_series):
    intervals = []
    in_run = False
    start = None
    for i, value in enumerate(binary_series):
        if value == 1 and not in_run:
            start = i
            in_run = True
        elif value == 0 and in_run:
            intervals.append((start, i - 1, i - start))
            in_run = False
    if in_run:
        intervals.append((start, len(binary_series) - 1, len(binary_series) - start))
    return intervals

# 4. Чтение EDF-файла
raw = mne.io.read_raw_edf(edf_path, preload=True, verbose="ERROR")
sfreq = raw.info["sfreq"]
duration_sec = raw.n_times / sfreq
print(f"\nЧастота дискретизации: {sfreq} Гц")
print(f"Длительность записи: {duration_sec:.1f} с")
print(f"Всего каналов: {len(raw.ch_names)}")
print("Каналы:", raw.ch_names)

# Разделение каналов
eeg_channels = [ch for ch in raw.ch_names if ch.startswith("EEG")]
non_eeg_channels = [ch for ch in raw.ch_names if not ch.startswith("EEG")]
print(f"EEG-каналов: {len(eeg_channels)}")
print("Не-EEG каналы:", non_eeg_channels)

# Обрезаем аннотационный ряд до длительности записи
ann_series = ann_series.iloc[:int(duration_sec)].reset_index(drop=True)

# 5. Определение всех приступных интервалов и выбор основного (самого длинного)
all_intervals = get_intervals(ann_series.values)
intervals_df = pd.DataFrame(all_intervals, columns=["start_sec", "end_sec", "duration_sec"])
intervals_df = intervals_df.sort_values(["duration_sec", "start_sec"], ascending=[False, True]).reset_index(drop=True)

print("\nНайденные приступные интервалы (первые 10):")
print(intervals_df.head(10).to_string(index=False))

seizure_start, seizure_end, seizure_duration = intervals_df.iloc[0]
print(f"\nВыбран основной интервал: {seizure_start}–{seizure_end} с (длительность {seizure_duration} с)")

# 6. График многоканальной ЭЭГ в начале приступа (первые 20 секунд)
raw_eeg = raw.copy().pick(eeg_channels)
data_uv = raw_eeg.get_data() * 1e6  # перевод в мкВ

plot_start_sec = int(seizure_start)
plot_end_sec = min(int(seizure_start) + 20, int(seizure_end))

start_idx = int(plot_start_sec * sfreq)
end_idx = int(plot_end_sec * sfreq)

segment = data_uv[:, start_idx:end_idx]
time_segment = np.arange(start_idx, end_idx) / sfreq

offset_step = 300  # шаг смещения по вертикали
plt.figure(figsize=(15, 8))
for i in range(segment.shape[0]):
    plt.plot(time_segment, segment[i] + i * offset_step, linewidth=0.8)

plt.yticks(
    [i * offset_step for i in range(len(eeg_channels))],
    [ch.replace("EEG ", "") for ch in eeg_channels]
)
plt.xlabel("Время, с")
plt.ylabel("Каналы EEG")
plt.title(f"Многоканальная ЭЭГ в начале приступа ({plot_start_sec}–{plot_end_sec} с)")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(WORK_DIR / "eeg2_multichannel_seizure.png", dpi=300, bbox_inches="tight")
plt.show()

# 7. Усреднение всех EEG-каналов в один сигнал
avg_signal_uv = data_uv.mean(axis=0)

# 8. Низкочастотная фильтрация Баттерворта (4-й порядок, срез 60 Гц)
sos = signal.butter(4, 60, btype="lowpass", fs=sfreq, output="sos")
avg_signal_filt_uv = signal.sosfiltfilt(sos, avg_signal_uv)

# 9. Выделение только приступного интервала для анализа
seizure_start_idx = int(seizure_start * sfreq)
seizure_end_idx = int(seizure_end * sfreq)

avg_seizure_uv = avg_signal_filt_uv[seizure_start_idx:seizure_end_idx]
time_seizure = np.arange(seizure_start_idx, seizure_end_idx) / sfreq

# График усреднённого отфильтрованного сигнала во время приступа
plt.figure(figsize=(15, 5))
plt.plot(time_seizure, avg_seizure_uv, color="black", linewidth=0.9)
plt.xlabel("Время, с")
plt.ylabel("Амплитуда, мкВ")
plt.title(f"Усреднённый EEG-сигнал после НЧ-фильтрации (≤60 Гц), интервал {seizure_start}–{seizure_end} с")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK_DIR / "eeg2_avg_filtered_seizure.png", dpi=300, bbox_inches="tight")
plt.show()

# 10. Спектрограмма (STFT)
frequencies, times_spec, Sxx = signal.spectrogram(
    avg_seizure_uv,
    fs=sfreq,
    window="hann",
    nperseg=512,
    noverlap=384,
    scaling="density",
    mode="psd"
)

freq_mask = frequencies <= 60
frequencies = frequencies[freq_mask]
Sxx = Sxx[freq_mask, :]

mean_psd = Sxx.mean(axis=1)
dominant_freq = frequencies[np.argmax(mean_psd)]
print(f"\nДоминирующая частота по средней спектральной мощности: {dominant_freq:.2f} Гц")

plt.figure(figsize=(15, 6))
plt.pcolormesh(times_spec + seizure_start, frequencies, 10 * np.log10(Sxx + 1e-12), shading="gouraud")
plt.colorbar(label="Мощность, дБ")
plt.xlabel("Время, с")
plt.ylabel("Частота, Гц")
plt.title("Спектрограмма усреднённого отфильтрованного EEG-сигнала")
plt.ylim(0, 60)
plt.tight_layout()
plt.savefig(WORK_DIR / "eeg2_spectrogram.png", dpi=300, bbox_inches="tight")
plt.show()

# 11. Вейвлет-скейлограмма (комплексный Morlet)
freqs = np.arange(1, 61)
n_cycles = np.maximum(freqs / 2, 2)

power = mne.time_frequency.tfr_array_morlet(
    avg_seizure_uv[np.newaxis, np.newaxis, :],
    sfreq=sfreq,
    freqs=freqs,
    n_cycles=n_cycles,
    output="power",
    decim=2,
    n_jobs=1,
    verbose="ERROR"
)[0, 0]

time_wavelet = time_seizure[::2]  # учтён децимированный временной ряд

plt.figure(figsize=(15, 6))
plt.pcolormesh(time_wavelet, freqs, power, shading="gouraud")
plt.colorbar(label="Мощность")
plt.xlabel("Время, с")
plt.ylabel("Частота, Гц")
plt.title("Вейвлет-скейлограмма усреднённого отфильтрованного EEG-сигнала")
plt.ylim(1, 60)
plt.tight_layout()
plt.savefig(WORK_DIR / "eeg2_scalogram.png", dpi=300, bbox_inches="tight")
plt.show()

# 12. Количественный анализ пиков по частотным диапазонам
band_definitions = {
    "delta_1_4_Hz": (1, 4),
    "theta_4_8_Hz": (4, 8),
    "alpha_8_13_Hz": (8, 13),
    "beta_13_30_Hz": (13, 30),
    "gamma_30_60_Hz": (30, 60),
}

band_peaks = []
for band_name, (f_low, f_high) in band_definitions.items():
    mask = (freqs >= f_low) & (freqs < f_high)
    band_series = power[mask].mean(axis=0)
    peak_idx = int(np.argmax(band_series))
    band_peaks.append({
        "band": band_name,
        "peak_time_sec": float(time_wavelet[peak_idx]),
        "peak_power": float(band_series[peak_idx])
    })

band_peaks_df = pd.DataFrame(band_peaks)
print("\nПиковая мощность по частотным диапазонам (вейвлет-анализ):")
print(band_peaks_df.to_string(index=False))

# 13. Сохранение сводных метрик
summary_metrics = pd.DataFrame([{
    "record_name": record_name,
    "record_id": record_id,
    "sampling_rate_hz": float(sfreq),
    "duration_sec": float(duration_sec),
    "total_channels": int(len(raw.ch_names)),
    "eeg_channels": int(len(eeg_channels)),
    "chosen_seizure_start_sec": int(seizure_start),
    "chosen_seizure_end_sec": int(seizure_end),
    "chosen_seizure_duration_sec": int(seizure_duration),
    "dominant_frequency_hz": float(dominant_freq),
    "filtered_signal_mean_uv": float(avg_seizure_uv.mean()),
    "filtered_signal_std_uv": float(avg_seizure_uv.std()),
    "filtered_signal_min_uv": float(avg_seizure_uv.min()),
    "filtered_signal_max_uv": float(avg_seizure_uv.max())
}])

summary_metrics.to_csv(WORK_DIR / "experiment3_metrics.csv", index=False)
intervals_df.to_csv(WORK_DIR / "seizure_intervals_eeg2.csv", index=False)
band_peaks_df.to_csv(WORK_DIR / "wavelet_band_peaks.csv", index=False)

print("\nМетрики эксперимента:")
print(summary_metrics.to_string(index=False))
print("\nФайлы сохранены в рабочую папку:")
print("- eeg2_multichannel_seizure.png")
print("- eeg2_avg_filtered_seizure.png")
print("- eeg2_spectrogram.png")
print("- eeg2_scalogram.png")
print("- experiment3_metrics.csv")
print("- seizure_intervals_eeg2.csv")
print("- wavelet_band_peaks.csv")

Рабочая папка: d:\MASTER\Term4\2\Seminar_GHADEER\Lab03\LAB3

Запись: eeg2, ID = 2
ID                                                  2
EEG file                                         eeg2
Diagnosis                                 prematurity
Number of Reviewers Annotating Seizure              1
Name: 1, dtype: object

Частота дискретизации: 256.0 Гц
Длительность записи: 3761.0 с
Всего каналов: 21
Каналы: ['EEG Fp1-Ref', 'EEG Fp2-Ref', 'EEG F3-Ref', 'EEG F4-Ref', 'EEG F7-Ref', 'EEG F8-Ref', 'EEG Fz-Ref', 'EEG C3-Ref', 'EEG C4-Ref', 'EEG Cz-Ref', 'EEG T3-Ref', 'EEG T5-Ref', 'EEG T4-Ref', 'EEG T6-Ref', 'EEG P3-Ref', 'EEG P4-Ref', 'EEG Pz-Ref', 'EEG O1-Ref', 'EEG O2-Ref', 'ECG EKG', 'Resp Effort']
EEG-каналов: 19
Не-EEG каналы: ['ECG EKG', 'Resp Effort']

Найденные приступные интервалы (первые 10):
Empty DataFrame
Columns: [start_sec, end_sec, duration_sec]
Index: []


IndexError: single positional indexer is out-of-bounds